In [ ]:
# ============================================================
# CELL 1: All imports for the URL Pipeline
# ============================================================

import requests
import joblib
import pandas as pd
import numpy as np
import urllib.parse
import socket
import ssl
import whois          
import time
import warnings
warnings.filterwarnings('ignore')

print(" All imports successful!")

In [ ]:
# ============================================================
# CELL 2: Load the saved GB model from Phase 3
# ============================================================

gb_model      = joblib.load('../models/gradient_boosting_model.pkl')
feature_names = joblib.load('../models/feature_names.pkl')

print(" Model loaded successfully!")
print(f"   Model type    : {type(gb_model).__name__}")
print(f"   Features count: {len(feature_names)}")
print(f"\n Expected features:")
for i, f in enumerate(feature_names, 1):
    print(f"   {i:2}. {f}")

In [ ]:
# ============================================================
# CELL 3: Validate the short URL before doing anything else
#         Paper Section 3.1 Step 2 — check URL validity
# ============================================================

def is_valid_short_url(url):
    """
    Checks if the input looks like a valid short URL.
    The paper checks:
      - Starts with HTTPS
      - Length between 20 and 25 characters
    We also add basic format checking.
    """
    results = {
        'is_valid'      : False,
        'has_https'     : False,
        'length_ok'     : False,
        'url_length'    : len(url),
        'error_message' : None
    }
    
    # Check 1: Must start with http or https
    if not url.startswith(('http://', 'https://')):
        results['error_message'] = "URL must start with http:// or https://"
        return results
    
    results['has_https'] = url.startswith('https://')
    
    # Check 2: Length check (paper says 20-25 for short URLs)
    # We're slightly lenient here — some shorteners go up to 30
    if 15 <= len(url) <= 35:
        results['length_ok'] = True
    else:
        results['error_message'] = f"URL length {len(url)} seems unusual for a short URL"
    
    # If both checks pass
    if results['has_https'] or url.startswith('http://'):
        results['is_valid'] = True
        
    return results


# --- Test it ---
test_urls = [
    "https://bit.ly/3xYz123",
    "https://tinyurl.com/abc123",
    "not-a-url-at-all",
    "http://cutt.ly/xyz"
]

print("URL VALIDATION TESTS")
print("=" * 50)
for url in test_urls:
    result = is_valid_short_url(url)
    status = " Valid" if result['is_valid'] else " Invalid"
    print(f"{status} | Length: {result['url_length']:2} | {url}")

In [ ]:
# ============================================================
# CELL 4: Expand the short URL to reveal true destination
#         Paper Section 3.1 Step 3 — unshorten the URL
# ============================================================

def unshorten_url(short_url, timeout=10):
    """
    Follows all redirects of a short URL and returns
    the final real destination URL.
    
    Example:
      Input : https://bit.ly/3xYz123
      Output: https://www.some-real-website.com/page/article
    """
    result = {
        'original_url'  : short_url,
        'expanded_url'  : None,
        'redirect_count': 0,
        'success'       : False,
        'error'         : None
    }
    
    try:
        headers = {
            # Pretend to be a real browser so servers don't block us
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        # allow_redirects=True means requests will follow ALL redirects
        # stream=True means don't download the whole page, just headers
        response = requests.get(
            short_url,
            allow_redirects=True,
            timeout=timeout,
            headers=headers,
            stream=True
        )
        
        result['expanded_url']   = response.url
        result['redirect_count'] = len(response.history)
        result['success']        = True
        
    except requests.exceptions.Timeout:
        result['error'] = "Timeout — URL took too long to respond"
    except requests.exceptions.ConnectionError:
        result['error'] = "Connection error — could not reach URL"
    except requests.exceptions.InvalidURL:
        result['error'] = "Invalid URL format"
    except Exception as e:
        result['error'] = f"Unexpected error: {str(e)}"
    
    return result


# --- Test it ---
print("URL UNSHORTENING TEST")
print("=" * 55)

# Using a safe known URL for testing
test_url = "https://tinyurl.com/wikipedia-en"
print(f"Testing with: {test_url}")
print("Please wait...")

result = unshorten_url(test_url)

if result['success']:
    print(f"\n Success!")
    print(f"   Original URL  : {result['original_url']}")
    print(f"   Expanded URL  : {result['expanded_url']}")
    print(f"   Redirects     : {result['redirect_count']}")
else:
    print(f"\n  {result['error']}")
    print("(This is okay — we'll handle errors gracefully)")

In [ ]:
# ============================================================
# CELL 5: Check the expanded URL against PhishTank database
#         Paper Section 3.1 Step 4 — blacklist check
# ============================================================

import os
import csv

def download_phishtank_db():
    """
    Downloads the PhishTank database CSV file.
    PhishTank provides a free offline CSV of all verified phishing URLs.
    We check our URL against this list.
    """
    db_path = '../data/raw/phishtank_db.csv'
    
    # If already downloaded, don't download again
    if os.path.exists(db_path):
        print(f" PhishTank DB already exists at {db_path}")
        return db_path
    
    print(" Downloading PhishTank database...")
    print("   (This is a large file — may take 1-2 minutes)")
    
    url = "http://data.phishtank.com/data/online-valid.csv"
    
    try:
        headers = {'User-Agent': 'phishtank/student-project'}
        response = requests.get(url, headers=headers, timeout=60)
        
        if response.status_code == 200:
            with open(db_path, 'wb') as f:
                f.write(response.content)
            print(f" Downloaded successfully! Saved to {db_path}")
            return db_path
        else:
            print(f" Server returned status {response.status_code}")
            return None
            
    except Exception as e:
        print(f" Could not download: {e}")
        print("   We'll use ML-only detection as fallback")
        return None


def check_phishtank(url, db_path):
    """
    Checks if a URL appears in the PhishTank blacklist CSV.
    Returns True if phishing, False if not found (assumed safe).
    """
    result = {
        'checked'   : False,
        'is_phishing': False,
        'match_found': False,
        'error'     : None
    }
    
    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available — skipping blacklist check"
        return result
    
    try:
        # Extract just the domain for comparison
        # e.g. https://www.evil-site.com/page → evil-site.com
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace('www.', '')
        
        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '')
                
                # Check if domain matches
                if url_domain and url_domain in phish_domain:
                    result['is_phishing'] = True
                    result['match_found'] = True
                    break
        
        result['checked'] = True
        
    except Exception as e:
        result['error'] = f"Error reading PhishTank DB: {str(e)}"
    
    return result


# --- Download the database ---
db_path = download_phishtank_db()

In [ ]:
# ============================================================
# CELL 6: Extract the 30 features from any URL
#         This is the most complex part — maps a real URL
#         to the same feature format our model was trained on
# ============================================================

def extract_features(url):
    """
    Extracts all 30 features from a URL.
    Features match the UCI phishing dataset exactly.
    Values: 1 (phishing indicator), -1 (legitimate indicator), 0 (suspicious)
    """
    features = {}
    
    try:
        parsed     = urllib.parse.urlparse(url)
        domain     = parsed.netloc.lower().replace('www.', '')
        full_url   = url.lower()
        path       = parsed.path.lower()
        
        # --------------------------------------------------
        # FEATURE 1: having_IP_Address
        # Does URL use an IP address instead of domain name?
        # --------------------------------------------------
        try:
            socket.inet_aton(domain.split(':')[0])
            features['having_IP_Address'] = 1   # IP found = phishing
        except:
            features['having_IP_Address'] = -1  # domain name = legitimate
        
        # --------------------------------------------------
        # FEATURE 2: URL_Length
        # Long URLs are suspicious
        # --------------------------------------------------
        url_len = len(url)
        if url_len < 54:
            features['URL_Length'] = -1
        elif url_len <= 75:
            features['URL_Length'] = 0
        else:
            features['URL_Length'] = 1
        
        # --------------------------------------------------
        # FEATURE 3: Shortining_Service (note paper's spelling)
        # Is this URL from a known shortening service?
        # --------------------------------------------------
        shorteners = ['bit.ly', 'tinyurl', 'goo.gl', 'ow.ly',
                      'cutt.ly', 't.co', 'short.io', 'rb.gy',
                      'is.gd', 'buff.ly', 'tiny.cc', 'tr.im']
        features['Shortining_Service'] = 1 if any(s in full_url for s in shorteners) else -1
        
        # --------------------------------------------------
        # FEATURE 4: having_At_Symbol
        # @ in URL = browser ignores everything before it
        # --------------------------------------------------
        features['having_At_Symbol'] = 1 if '@' in url else -1
        
        # --------------------------------------------------
        # FEATURE 5: double_slash_redirecting
        # // after the protocol position = suspicious redirect
        # --------------------------------------------------
        features['double_slash_redirecting'] = 1 if '//' in url[7:] else -1
        
        # --------------------------------------------------
        # FEATURE 6: Prefix_Suffix
        # Hyphen in domain = phishing trick (google-login.com)
        # --------------------------------------------------
        features['Prefix_Suffix'] = 1 if '-' in domain else -1
        
        # --------------------------------------------------
        # FEATURE 7: having_Sub_Domain
        # Count dots in domain — more dots = more subdomains
        # --------------------------------------------------
        dot_count = domain.count('.')
        if dot_count == 1:
            features['having_Sub_Domain'] = -1
        elif dot_count == 2:
            features['having_Sub_Domain'] = 0
        else:
            features['having_Sub_Domain'] = 1
        
        # --------------------------------------------------
        # FEATURE 8: SSLfinal_State
        # Does site use HTTPS with valid certificate?
        # --------------------------------------------------
        if url.startswith('https://'):
            try:
                context = ssl.create_default_context()
                conn    = context.wrap_socket(
                    socket.socket(socket.AF_INET),
                    server_hostname=domain.split(':')[0]
                )
                conn.settimeout(5)
                conn.connect((domain.split(':')[0], 443))
                conn.close()
                features['SSLfinal_State'] = 1   # valid SSL = legitimate
            except:
                features['SSLfinal_State'] = 0
        else:
            features['SSLfinal_State'] = -1      # no HTTPS = phishing
        
        # --------------------------------------------------
        # FEATURE 9: Domain_registeration_length
        # Try to get domain registration length via WHOIS
        # --------------------------------------------------
        try:
            w          = whois.whois(domain)
            expiry     = w.expiration_date
            creation   = w.creation_date
            if isinstance(expiry, list):
                expiry = expiry[0]
            if isinstance(creation, list):
                creation = creation[0]
            if expiry and creation:
                reg_length = (expiry - creation).days
                features['Domain_registeration_length'] = -1 if reg_length > 365 else 1
            else:
                features['Domain_registeration_length'] = 0
        except:
            features['Domain_registeration_length'] = 0
        
        # --------------------------------------------------
        # FEATURE 10: Favicon
        # Assume favicon loads from same domain (we can't
        # render pages, so we use a neutral value)
        # --------------------------------------------------
        features['Favicon'] = 0
        
        # --------------------------------------------------
        # FEATURE 11: port
        # Non-standard port in URL = suspicious
        # --------------------------------------------------
        suspicious_ports = ['21','22','23','80','443','8080','8443','8888']
        if parsed.port:
            features['port'] = 1 if str(parsed.port) not in suspicious_ports else -1
        else:
            features['port'] = -1
        
        # --------------------------------------------------
        # FEATURE 12: HTTPS_token
        # "https" appearing IN the domain name = fake trust
        # e.g. https-google.com
        # --------------------------------------------------
        features['HTTPS_token'] = 1 if 'https' in domain else -1
        
        # --------------------------------------------------
        # FEATURES 13-16: Page content features
        # We can't render pages, so use neutral values
        # These would need a full browser to check properly
        # --------------------------------------------------
        features['Request_URL']      = 0
        features['URL_of_Anchor']    = 0
        features['Links_in_tags']    = 0
        features['SFH']              = 0
        
        # --------------------------------------------------
        # FEATURE 17: Submitting_to_email
        # mailto: in URL = old phishing trick
        # --------------------------------------------------
        features['Submitting_to_email'] = 1 if 'mailto:' in full_url else -1
        
        # --------------------------------------------------
        # FEATURE 18: Abnormal_URL
        # Does domain appear in the URL path? 
        # Phishing URLs often don't match their own domain
        # --------------------------------------------------
        features['Abnormal_URL'] = -1 if domain in full_url else 1
        
        # --------------------------------------------------
        # FEATURE 19: Redirect
        # We'll set this after unshortening (passed as param)
        # --------------------------------------------------
        features['Redirect'] = 0
        
        # --------------------------------------------------
        # FEATURES 20-23: JavaScript behavior features
        # Can't check without rendering — use neutral
        # --------------------------------------------------
        features['on_mouseover']  = 0
        features['RightClick']    = 0
        features['popUpWidnow']   = 0
        features['Iframe']        = 0
        
        # --------------------------------------------------
        # FEATURE 24: age_of_domain
        # How old is the domain? New = suspicious
        # --------------------------------------------------
        try:
            w        = whois.whois(domain)
            creation = w.creation_date
            if isinstance(creation, list):
                creation = creation[0]
            if creation:
                from datetime import datetime
                age_days = (datetime.now() - creation).days
                features['age_of_domain'] = -1 if age_days > 180 else 1
            else:
                features['age_of_domain'] = 0
        except:
            features['age_of_domain'] = 0
        
        # --------------------------------------------------
        # FEATURE 25: DNSRecord
        # Does domain have a DNS record?
        # --------------------------------------------------
        try:
            socket.gethostbyname(domain.split(':')[0])
            features['DNSRecord'] = -1   # DNS exists = legitimate
        except:
            features['DNSRecord'] = 1    # No DNS = phishing
        
        # --------------------------------------------------
        # FEATURES 26-30: External service features
        # These require APIs (Alexa, Google PageRank etc)
        # We use neutral values as fallback
        # --------------------------------------------------
        features['web_traffic']            = 0
        features['Page_Rank']              = 0
        features['Google_Index']           = 0
        features['Links_pointing_to_page'] = 0
        features['Statistical_report']     = 0
        
    except Exception as e:
        print(f"  Feature extraction error: {e}")
        # Return all neutral values if something breaks
        for fname in feature_names:
            if fname not in features:
                features[fname] = 0
    
    # Make sure all features are present in correct order
    final_features = {fname: features.get(fname, 0) for fname in feature_names}
    
    return final_features


# --- Test it ---
print("FEATURE EXTRACTION TEST")
print("=" * 55)
test_url      = "https://www.google.com"
test_features = extract_features(test_url)
print(f"URL: {test_url}")
print(f"\nExtracted {len(test_features)} features:")
for fname, fval in test_features.items():
    meaning = {1: "  phishing", -1: " legit", 0: " neutral"}
    print(f"  {fname:<35}: {fval:2}  {meaning.get(fval,'')}")

In [ ]:
# ============================================================
# CELL 7: Run the saved GB model on extracted features
#         Returns prediction + confidence score
# ============================================================

def predict_url(features_dict):
    """
    Takes the feature dictionary and runs it through
    our saved Gradient Boosting model.
    Returns prediction and confidence score.
    """
    # Convert dict to DataFrame with correct column order
    features_df = pd.DataFrame([features_dict])[feature_names]
    
    # Get prediction: 1=phishing, -1=legitimate
    prediction = gb_model.predict(features_df)[0]
    
    # Get probability scores [prob_legitimate, prob_phishing]
    probabilities = gb_model.predict_proba(features_df)[0]
    
    # Find which class is which
    classes           = gb_model.classes_.tolist()
    phishing_idx      = classes.index(1)
    legitimate_idx    = classes.index(-1)
    
    phishing_prob   = probabilities[phishing_idx]
    legitimate_prob = probabilities[legitimate_idx]
    
    return {
        'prediction'    : prediction,
        'is_phishing'   : prediction == 1,
        'phishing_prob' : round(phishing_prob * 100, 1),
        'legitimate_prob': round(legitimate_prob * 100, 1),
        'confidence'    : round(max(phishing_prob, legitimate_prob) * 100, 1)
    }


# --- Test it ---
print("ML PREDICTOR TEST")
print("=" * 55)

# Test with google.com features we extracted above
ml_result = predict_url(test_features)

print(f"URL tested     : https://www.google.com")
print(f"Prediction     : {' PHISHING' if ml_result['is_phishing'] else ' LEGITIMATE'}")
print(f"Phishing prob  : {ml_result['phishing_prob']}%")
print(f"Legitimate prob: {ml_result['legitimate_prob']}%")
print(f"Confidence     : {ml_result['confidence']}%")

In [ ]:
# ============================================================
# CELL 8: Put ALL steps together into one clean function
#         This is the heart of our entire system
# ============================================================

def check_url(short_url, db_path=None, verbose=True):
    """
    Complete Phishing URL Detection System pipeline:
    Step 1: Validate the URL
    Step 2: Unshorten it
    Step 3: Check PhishTank blacklist
    Step 4: Extract 30 features
    Step 5: Run GB model
    Step 6: Return final verdict
    """
    
    if verbose:
        print("\n" + "=" * 60)
        print(" PHISHING SHORT URL DETECTION SYSTEM (Phishing URL Detection System)")
        print("=" * 60)
        print(f"Input URL: {short_url}")
        print("-" * 60)
    
    report = {
        'input_url'      : short_url,
        'expanded_url'   : None,
        'is_valid'       : False,
        'phishtank_result': None,
        'ml_result'      : None,
        'final_verdict'  : None,
        'error'          : None
    }
    
    # ── STEP 1: Validate ──────────────────────────────────
    if verbose: print("\n Step 1: Validating URL...")
    validation = is_valid_short_url(short_url)
    
    if not validation['is_valid']:
        report['error']         = validation['error_message']
        report['final_verdict'] = 'INVALID URL'
        if verbose:
            print(f"    Invalid: {validation['error_message']}")
        return report
    
    if verbose: print(f"    Valid | Length: {validation['url_length']} chars")
    report['is_valid'] = True
    
    # ── STEP 2: Unshorten ─────────────────────────────────
    if verbose: print("\n Step 2: Unshortening URL...")
    unshorten_result = unshorten_url(short_url)
    
    if not unshorten_result['success']:
        # Use original URL if we can't expand it
        expanded = short_url
        if verbose:
            print(f"     Could not expand: {unshorten_result['error']}")
            print(f"   ️  Using original URL for analysis")
    else:
        expanded = unshorten_result['expanded_url']
        if verbose:
            print(f"    Expanded URL  : {expanded}")
            print(f"    Redirects     : {unshorten_result['redirect_count']}")
    
    report['expanded_url'] = expanded
    
    # ── STEP 3: PhishTank Check ───────────────────────────
    if verbose: print("\n️  Step 3: Checking PhishTank blacklist...")
    pt_result = check_phishtank(expanded, db_path)
    
    if pt_result['error']:
        if verbose: print(f"     {pt_result['error']}")
        report['phishtank_result'] = 'UNAVAILABLE'
    elif pt_result['is_phishing']:
        if verbose: print(f"    FOUND IN PHISHTANK BLACKLIST!")
        report['phishtank_result'] = 'PHISHING'
    else:
        if verbose: print(f"    Not found in PhishTank — appears safe")
        report['phishtank_result'] = 'SAFE'
    
    # ── STEP 4 + 5: Feature Extraction + ML ───────────────
    if verbose: print("\n Step 4: Extracting features + running ML model...")
    features   = extract_features(expanded)
    ml_result  = predict_url(features)
    report['ml_result'] = ml_result
    
    if verbose:
        verdict_str = " PHISHING" if ml_result['is_phishing'] else " LEGITIMATE"
        print(f"   ML Verdict  : {verdict_str}")
        print(f"   Confidence  : {ml_result['confidence']}%")
    
    # ── STEP 6: Final Verdict ─────────────────────────────
    # Logic: if EITHER PhishTank OR ML says phishing → PHISHING
    pt_phishing = report['phishtank_result'] == 'PHISHING'
    ml_phishing = ml_result['is_phishing']
    
    if pt_phishing and ml_phishing:
        report['final_verdict'] = 'PHISHING'
        verdict_note            = "(Both PhishTank AND ML flagged this)"
    elif pt_phishing:
        report['final_verdict'] = 'PHISHING'
        verdict_note            = "(PhishTank blacklist flagged this)"
    elif ml_phishing:
        report['final_verdict'] = 'PHISHING'
        verdict_note            = "(ML model flagged this)"
    else:
        report['final_verdict'] = 'SAFE'
        verdict_note            = "(Passed both PhishTank and ML checks)"
    
    if verbose:
        print("\n" + "=" * 60)
        if report['final_verdict'] == 'PHISHING':
            print(f" FINAL VERDICT: PHISHING — ACCESS BLOCKED")
        else:
            print(f" FINAL VERDICT: SAFE — OK to proceed")
        print(f"   {verdict_note}")
        print("=" * 60)
    
    return report


print(" Pipeline function defined and ready!")

In [ ]:
# ============================================================
# CELL 9: Test on 10 real URLs — mix of safe and suspicious
# ============================================================

# Safe short URLs pointing to known legitimate sites
test_urls = [
    "https://tinyurl.com/wikipedia-en",
    "https://bit.ly/3google",
    "https://cutt.ly/github",
    "https://tinyurl.com/python-docs",
    "https://rb.gy/stackoverflow",
]

results_log = []

print("TESTING 5 URLs THROUGH COMPLETE PIPELINE")
print("=" * 60)

for url in test_urls:
    print(f"\n{'─'*60}")
    result = check_url(url, db_path=db_path, verbose=True)
    results_log.append({
        'Input URL'     : result['input_url'],
        'Expanded URL'  : result['expanded_url'],
        'PhishTank'     : result['phishtank_result'],
        'ML Verdict'    : ' Phishing' if result.get('ml_result') and 
                          result['ml_result']['is_phishing'] else ' Safe',
        'Confidence'    : f"{result['ml_result']['confidence']}%" 
                          if result.get('ml_result') else 'N/A',
        'Final Verdict' : result['final_verdict']
    })
    time.sleep(1)  # be polite — don't hammer servers

# Summary table
print("\n\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
summary_df = pd.DataFrame(results_log)
display(summary_df)

In [ ]:
# ============================================================
# CELL 10: Save all pipeline functions to a .py file
#          So we can import them cleanly in Phase 5 and 6
# ============================================================

pipeline_code = '''
import requests
import joblib
import pandas as pd
import numpy as np
import urllib.parse
import socket
import ssl
import whois
import time
import csv
import os
import warnings
warnings.filterwarnings("ignore")

# Load model and feature names
BASE_DIR      = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH    = os.path.join(BASE_DIR, "../models/gradient_boosting_model.pkl")
FEATURES_PATH = os.path.join(BASE_DIR, "../models/feature_names.pkl")
PHISHTANK_DB  = os.path.join(BASE_DIR, "../data/raw/phishtank_db.csv")

gb_model      = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURES_PATH)


def is_valid_short_url(url):
    result = {"is_valid": False, "error_message": None, "url_length": len(url)}
    if not url.startswith(("http://", "https://")):
        result["error_message"] = "URL must start with http:// or https://"
        return result
    result["is_valid"] = True
    return result


def unshorten_url(short_url, timeout=10):
    result = {"original_url": short_url, "expanded_url": None,
              "redirect_count": 0, "success": False, "error": None}
    try:
        headers  = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(short_url, allow_redirects=True,
                                timeout=timeout, headers=headers, stream=True)
        result["expanded_url"]   = response.url
        result["redirect_count"] = len(response.history)
        result["success"]        = True
    except requests.exceptions.Timeout:
        result["error"] = "Timeout"
    except Exception as e:
        result["error"] = str(e)
    return result


def check_phishtank(url, db_path=PHISHTANK_DB):
    result = {"checked": False, "is_phishing": False, "error": None}
    if not db_path or not os.path.exists(db_path):
        result["error"] = "PhishTank DB not available"
        return result
    try:
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace("www.", "")
        with open(db_path, "r", encoding="utf-8", errors="ignore") as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get("url", "").lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace("www.", "")
                if url_domain and url_domain in phish_domain:
                    result["is_phishing"] = True
                    break
        result["checked"] = True
    except Exception as e:
        result["error"] = str(e)
    return result


def extract_features(url):
    features = {}
    try:
        parsed   = urllib.parse.urlparse(url)
        domain   = parsed.netloc.lower().replace("www.", "")
        full_url = url.lower()
        try:
            socket.inet_aton(domain.split(":")[0])
            features["having_IP_Address"] = 1
        except:
            features["having_IP_Address"] = -1
        url_len = len(url)
        features["URL_Length"] = -1 if url_len < 54 else (0 if url_len <= 75 else 1)
        shorteners = ["bit.ly","tinyurl","goo.gl","ow.ly","cutt.ly","t.co","short.io","rb.gy"]
        features["Shortining_Service"]         = 1 if any(s in full_url for s in shorteners) else -1
        features["having_At_Symbol"]           = 1 if "@" in url else -1
        features["double_slash_redirecting"]   = 1 if "//" in url[7:] else -1
        features["Prefix_Suffix"]              = 1 if "-" in domain else -1
        dot_count = domain.count(".")
        features["having_Sub_Domain"] = -1 if dot_count == 1 else (0 if dot_count == 2 else 1)
        features["SSLfinal_State"]             = 1 if url.startswith("https://") else -1
        features["Domain_registeration_length"] = 0
        features["Favicon"]                    = 0
        features["port"]                       = -1 if not parsed.port else 1
        features["HTTPS_token"]                = 1 if "https" in domain else -1
        features["Request_URL"]                = 0
        features["URL_of_Anchor"]              = 0
        features["Links_in_tags"]              = 0
        features["SFH"]                        = 0
        features["Submitting_to_email"]        = 1 if "mailto:" in full_url else -1
        features["Abnormal_URL"]               = -1 if domain in full_url else 1
        features["Redirect"]                   = 0
        features["on_mouseover"]               = 0
        features["RightClick"]                 = 0
        features["popUpWidnow"]                = 0
        features["Iframe"]                     = 0
        features["age_of_domain"]              = 0
        try:
            socket.gethostbyname(domain.split(":")[0])
            features["DNSRecord"] = -1
        except:
            features["DNSRecord"] = 1
        features["web_traffic"]            = 0
        features["Page_Rank"]              = 0
        features["Google_Index"]           = 0
        features["Links_pointing_to_page"] = 0
        features["Statistical_report"]     = 0
    except Exception:
        pass
    return {fname: features.get(fname, 0) for fname in feature_names}


def predict_url(features_dict):
    features_df   = pd.DataFrame([features_dict])[feature_names]
    prediction    = gb_model.predict(features_df)[0]
    probabilities = gb_model.predict_proba(features_df)[0]
    classes       = gb_model.classes_.tolist()
    phishing_prob = probabilities[classes.index(1)]
    return {
        "prediction"     : prediction,
        "is_phishing"    : prediction == 1,
        "phishing_prob"  : round(phishing_prob * 100, 1),
        "legitimate_prob": round((1 - phishing_prob) * 100, 1),
        "confidence"     : round(max(phishing_prob, 1 - phishing_prob) * 100, 1)
    }


def check_url(short_url, db_path=PHISHTANK_DB, verbose=True):
    report = {"input_url": short_url, "expanded_url": None,
              "phishtank_result": None, "ml_result": None,
              "final_verdict": None, "error": None}
    validation = is_valid_short_url(short_url)
    if not validation["is_valid"]:
        report["error"]         = validation["error_message"]
        report["final_verdict"] = "INVALID URL"
        return report
    unshorten_result      = unshorten_url(short_url)
    expanded              = unshorten_result["expanded_url"] if unshorten_result["success"] else short_url
    report["expanded_url"] = expanded
    pt_result             = check_phishtank(expanded, db_path)
    report["phishtank_result"] = "PHISHING" if pt_result["is_phishing"] else (
                                 "UNAVAILABLE" if pt_result["error"] else "SAFE")
    features              = extract_features(expanded)
    ml_result             = predict_url(features)
    report["ml_result"]   = ml_result
    pt_phishing           = report["phishtank_result"] == "PHISHING"
    ml_phishing           = ml_result["is_phishing"]
    report["final_verdict"] = "PHISHING" if (pt_phishing or ml_phishing) else "SAFE"
    return report
'''

with open('../src/url_pipeline.py', 'w') as f:
    f.write(pipeline_code)

print(" Pipeline saved to src/url_pipeline.py")
print("   This file will be imported in Phase 5 and Phase 6")

In [ ]:
# ============================================================
# IMPROVED VERDICT LOGIC
# Adds confidence threshold so low-confidence predictions
# don't trigger false alarms
# ============================================================

def check_url_improved(short_url, db_path=None, verbose=True):
    """
    Improved version with confidence threshold.
    Only flags as PHISHING if model is reasonably confident.
    """
    
    if verbose:
        print("\n" + "=" * 60)
        print(" Phishing URL Detection System — IMPROVED DETECTION")
        print("=" * 60)
        print(f"Input URL: {short_url}")
        print("-" * 60)

    report = {
        'input_url'       : short_url,
        'expanded_url'    : None,
        'phishtank_result': None,
        'ml_result'       : None,
        'final_verdict'   : None,
        'verdict_reason'  : None,
        'error'           : None
    }

    # Step 1: Validate
    validation = is_valid_short_url(short_url)
    if not validation['is_valid']:
        report['error']         = validation['error_message']
        report['final_verdict'] = 'INVALID URL'
        return report

    # Step 2: Unshorten
    if verbose: print("\n Unshortening URL...")
    unshorten_result = unshorten_url(short_url)
    expanded = (unshorten_result['expanded_url'] 
                if unshorten_result['success'] else short_url)
    report['expanded_url'] = expanded
    if verbose: print(f"   Expanded: {expanded}")

    # Step 3: PhishTank
    if verbose: print("\n️  Checking PhishTank...")
    pt_result = check_phishtank(expanded, db_path)
    
    if pt_result.get('error'):
        report['phishtank_result'] = 'UNAVAILABLE'
        if verbose: print(f"     Unavailable")
    elif pt_result['is_phishing']:
        report['phishtank_result'] = 'PHISHING'
        if verbose: print(f"    Found in blacklist!")
    else:
        report['phishtank_result'] = 'SAFE'
        if verbose: print(f"    Not in blacklist")

    # Step 4+5: Features + ML
    if verbose: print("\n Running ML model...")
    features  = extract_features(expanded)
    ml_result = predict_url(features)
    report['ml_result'] = ml_result

    if verbose:
        print(f"   Phishing probability : {ml_result['phishing_prob']}%")
        print(f"   Legitimate probability: {ml_result['legitimate_prob']}%")
        print(f"   Confidence           : {ml_result['confidence']}%")

    # ── IMPROVED VERDICT LOGIC ─────────────────────────────
    # Key change: ML only triggers PHISHING if confidence > 70%
    # Below 70% = model is uncertain = don't block
    
    CONFIDENCE_THRESHOLD = 70.0  # you can tune this value

    pt_phishing = report['phishtank_result'] == 'PHISHING'
    ml_phishing = (ml_result['is_phishing'] and 
                   ml_result['phishing_prob'] >= CONFIDENCE_THRESHOLD)

    if pt_phishing and ml_phishing:
        report['final_verdict']  = 'PHISHING'
        report['verdict_reason'] = 'Flagged by BOTH PhishTank and ML model'
    elif pt_phishing:
        report['final_verdict']  = 'PHISHING'
        report['verdict_reason'] = 'Found in PhishTank blacklist'
    elif ml_phishing:
        report['final_verdict']  = 'PHISHING'
        report['verdict_reason'] = f'ML confidence: {ml_result["phishing_prob"]}% (above {CONFIDENCE_THRESHOLD}% threshold)'
    else:
        report['final_verdict']  = 'SAFE'
        if ml_result['phishing_prob'] < CONFIDENCE_THRESHOLD and ml_result['is_phishing']:
            report['verdict_reason'] = f'ML flagged but low confidence ({ml_result["phishing_prob"]}%) — treated as SAFE'
        else:
            report['verdict_reason'] = 'Passed both checks'

    # Print final verdict
    if verbose:
        print("\n" + "=" * 60)
        if report['final_verdict'] == 'PHISHING':
            print(" FINAL VERDICT: PHISHING — BLOCKED")
        else:
            print(" FINAL VERDICT: SAFE")
        print(f"   Reason: {report['verdict_reason']}")
        print("=" * 60)

    return report


# --- Test it on Google ---
print("Testing improved pipeline on known safe URLs:\n")
result = check_url_improved("https://www.google.com", db_path=db_path)

In [ ]:
# ============================================================
# RETEST all URLs with improved logic
# ============================================================

test_urls = [
    "https://tinyurl.com/wikipedia-en",
    "https://bit.ly/3google",
    "https://cutt.ly/github",
    "https://tinyurl.com/python-docs",
    "https://rb.gy/stackoverflow",
]

results_log = []

for url in test_urls:
    print(f"\n{'─'*60}")
    result = check_url_improved(url, db_path=db_path, verbose=True)
    results_log.append({
        'Input URL'    : result['input_url'],
        'PhishTank'    : result['phishtank_result'],
        'ML Prob'      : f"{result['ml_result']['phishing_prob']}%" 
                         if result.get('ml_result') else 'N/A',
        'Final Verdict': result['final_verdict'],
        'Reason'       : result['verdict_reason']
    })
    time.sleep(1)

print("\n\n" + "=" * 60)
print("IMPROVED RESULTS SUMMARY")
print("=" * 60)
display(pd.DataFrame(results_log))

In [ ]:
# ============================================================
# FIXED PhishTank checker — exact domain matching only
# Replace the loose substring match with strict exact match
# ============================================================

def check_phishtank_fixed(url, db_path=None):
    """
    Fixed version — only matches if domains are exactly equal.
    Prevents false positives like google.com matching 
    google-login.evil.com
    """
    result = {
        'checked'    : False,
        'is_phishing': False,
        'error'      : None
    }
    
    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available"
        return result
    
    try:
        parsed     = urllib.parse.urlparse(url)
        # Get exact domain only — no substrings
        url_domain = parsed.netloc.lower().replace('www.', '').strip()
        
        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '').strip()
                
                # STRICT: domains must match exactly
                # NOT substring — full domain equality only
                if url_domain and phish_domain and url_domain == phish_domain:
                    result['is_phishing'] = True
                    break
        
        result['checked'] = True
        
    except Exception as e:
        result['error'] = str(e)
    
    return result


# --- Test the fix on Google ---
print("Testing fixed PhishTank checker:")
print("=" * 45)

test_result = check_phishtank_fixed("https://www.google.com", db_path)
print(f"Google.com is phishing: {test_result['is_phishing']}")
print(f"Expected              : False")
print(f"Fix worked            : {not test_result['is_phishing']} ")

In [ ]:
# ============================================================
# Updated pipeline using the fixed PhishTank checker
# ============================================================

def check_url_final(short_url, db_path=None, verbose=True):
    """
    Final version with:
    1. Fixed PhishTank exact domain matching
    2. Confidence threshold for ML
    """
    
    if verbose:
        print("\n" + "=" * 60)
        print(" Phishing URL Detection System — FINAL DETECTION")
        print("=" * 60)
        print(f"Input URL: {short_url}")
        print("-" * 60)

    report = {
        'input_url'       : short_url,
        'expanded_url'    : None,
        'phishtank_result': None,
        'ml_result'       : None,
        'final_verdict'   : None,
        'verdict_reason'  : None,
        'error'           : None
    }

    # Step 1: Validate
    validation = is_valid_short_url(short_url)
    if not validation['is_valid']:
        report['error']         = validation['error_message']
        report['final_verdict'] = 'INVALID URL'
        return report

    # Step 2: Unshorten
    if verbose: print("\n Unshortening URL...")
    unshorten_result = unshorten_url(short_url)
    expanded = (unshorten_result['expanded_url']
                if unshorten_result['success'] else short_url)
    report['expanded_url'] = expanded
    if verbose: print(f"   Expanded: {expanded}")

    # Step 3: PhishTank — NOW USING FIXED VERSION
    if verbose: print("\n️  Checking PhishTank...")
    pt_result = check_phishtank_fixed(expanded, db_path)

    if pt_result.get('error'):
        report['phishtank_result'] = 'UNAVAILABLE'
        if verbose: print(f"     Unavailable")
    elif pt_result['is_phishing']:
        report['phishtank_result'] = 'PHISHING'
        if verbose: print(f"    Found in blacklist!")
    else:
        report['phishtank_result'] = 'SAFE'
        if verbose: print(f"    Not in blacklist")

    # Step 4+5: Features + ML
    if verbose: print("\n Running ML model...")
    features  = extract_features(expanded)
    ml_result = predict_url(features)
    report['ml_result'] = ml_result

    if verbose:
        print(f"   Phishing probability : {ml_result['phishing_prob']}%")
        print(f"   Legitimate probability: {ml_result['legitimate_prob']}%")
        print(f"   Confidence           : {ml_result['confidence']}%")

    # Step 6: Verdict with confidence threshold
    CONFIDENCE_THRESHOLD = 70.0

    pt_phishing = report['phishtank_result'] == 'PHISHING'
    ml_phishing = (ml_result['is_phishing'] and
                   ml_result['phishing_prob'] >= CONFIDENCE_THRESHOLD)

    if pt_phishing and ml_phishing:
        report['final_verdict']  = 'PHISHING'
        report['verdict_reason'] = 'Flagged by BOTH PhishTank and ML'
    elif pt_phishing:
        report['final_verdict']  = 'PHISHING'
        report['verdict_reason'] = 'Found in PhishTank blacklist'
    elif ml_phishing:
        report['final_verdict']  = 'PHISHING'
        report['verdict_reason'] = f'ML confidence {ml_result["phishing_prob"]}% above threshold'
    else:
        report['final_verdict']  = 'SAFE'
        report['verdict_reason'] = (
            f'ML flagged but low confidence ({ml_result["phishing_prob"]}%) — treated as SAFE'
            if ml_result['is_phishing']
            else 'Passed both checks'
        )

    if verbose:
        print("\n" + "=" * 60)
        if report['final_verdict'] == 'PHISHING':
            print(" FINAL VERDICT: PHISHING — BLOCKED")
        else:
            print(" FINAL VERDICT: SAFE")
        print(f"   Reason: {report['verdict_reason']}")
        print("=" * 60)

    return report


# --- Test on Google now ---
result = check_url_final("https://www.google.com", db_path=db_path)

In [ ]:
# CELL A — Redefine the fixed function (run this first)
def check_phishtank_fixed(url, db_path=None):
    result = {
        'checked'    : False,
        'is_phishing': False,
        'error'      : None
    }
    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available"
        return result
    try:
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace('www.', '').strip()
        
        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '').strip()
                
                # EXACT match only — no substrings
                if url_domain and phish_domain and url_domain == phish_domain:
                    result['is_phishing'] = True
                    break
        result['checked'] = True
    except Exception as e:
        result['error'] = str(e)
    return result

print("Fixed function defined!")

In [ ]:
# CELL B — Now test it properly
import urllib.parse
import csv

# Test 1: google.com should be SAFE
test1 = check_phishtank_fixed("https://www.google.com", db_path)
print(f"google.com    is phishing: {test1['is_phishing']} (expected: False)")

# Test 2: docs.google.com should be PHISHING (it genuinely is in database)
test2 = check_phishtank_fixed("https://docs.google.com/some-fake-page", db_path)
print(f"docs.google.com is phishing: {test2['is_phishing']} (expected: True)")

# Test 3: sites.google.com should be PHISHING
test3 = check_phishtank_fixed("https://sites.google.com/view/fake", db_path)
print(f"sites.google.com is phishing: {test3['is_phishing']} (expected: True)")

In [ ]:
# ============================================================
# PROPERLY FIXED PhishTank checker
# Matches on full URL comparison not just domain
# ============================================================

def check_phishtank_fixed(url, db_path=None):
    result = {
        'checked'    : False,
        'is_phishing': False,
        'error'      : None
    }
    
    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available"
        return result
    
    try:
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace('www.', '').strip()
        url_path   = parsed.path.lower().strip()
        
        # Whitelist of major trusted domains
        # We skip domain-only matching for these
        # and require full URL match instead
        trusted_domains = [
            'google.com', 'microsoft.com', 'apple.com',
            'amazon.com', 'facebook.com', 'twitter.com',
            'linkedin.com', 'youtube.com', 'github.com',
            'wikipedia.org', 'stackoverflow.com'
        ]
        
        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '').strip()
                phish_path   = phish_parsed.path.lower().strip()
                
                if url_domain in trusted_domains:
                    # For trusted domains match BOTH domain AND path exactly
                    # This catches google.com/url?q=evil but not google.com/
                    if url_domain == phish_domain and url_path == phish_path:
                        result['is_phishing'] = True
                        break
                else:
                    # For unknown domains exact domain match is enough
                    if url_domain and phish_domain and url_domain == phish_domain:
                        result['is_phishing'] = True
                        break
        
        result['checked'] = True
        
    except Exception as e:
        result['error'] = str(e)
    
    return result


# --- Test all 3 cases ---
print("FINAL PHISHTANK TESTS")
print("=" * 50)

test1 = check_phishtank_fixed("https://www.google.com", db_path)
print(f"google.com homepage  : {test1['is_phishing']} (expected: False) {'' if not test1['is_phishing'] else ''}")

test2 = check_phishtank_fixed("https://docs.google.com/some-fake-page", db_path)
print(f"docs.google.com fake : {test2['is_phishing']} (expected: True)  {'' if test2['is_phishing'] else ''}")

test3 = check_phishtank_fixed("https://www.google.com/url?q=https://evil.com", db_path)
print(f"google redirect evil : {test3['is_phishing']} (expected: True)  {'' if test3['is_phishing'] else ''}")

In [ ]:
# ============================================================
# UNIVERSAL FIX — Works for ALL domains
# No whitelist needed — detects redirect abuse pattern
# ============================================================

def check_phishtank_fixed(url, db_path=None):
    result = {
        'checked'    : False,
        'is_phishing': False,
        'error'      : None
    }
    
    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available"
        return result
    
    # These path patterns mean the domain is being abused
    # as a redirect — not that the domain itself is phishing
    redirect_abuse_patterns = [
        '/url?', '/url?q=', '/amp/', '/amp/s/',
        '/redirect', '/l?u=', '/link?'
    ]
    
    try:
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace('www.', '').strip()
        url_path   = parsed.path.lower().strip()
        url_query  = parsed.query.lower().strip()
        
        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '').strip()
                phish_path   = phish_parsed.path.lower().strip()
                phish_query  = phish_parsed.query.lower().strip()
                
                # Domains must match exactly — always
                if url_domain != phish_domain:
                    continue
                
                # Check if the PhishTank entry is a redirect abuse
                phish_is_redirect = any(
                    pattern in phish_path or pattern in ('?' + phish_query)
                    for pattern in redirect_abuse_patterns
                )
                
                if phish_is_redirect:
                    # For redirect abuse entries:
                    # Only match if our URL ALSO has redirect pattern
                    # google.com ≠ google.com/url?q=evil
                    our_is_redirect = any(
                        pattern in url_path or pattern in ('?' + url_query)
                        for pattern in redirect_abuse_patterns
                    )
                    if our_is_redirect:
                        result['is_phishing'] = True
                        break
                else:
                    # Normal phishing domain — domain match is enough
                    result['is_phishing'] = True
                    break
        
        result['checked'] = True
        
    except Exception as e:
        result['error'] = str(e)
    
    return result


# --- Test ALL cases ---
print("UNIVERSAL PHISHTANK TESTS")
print("=" * 55)

tests = [
    ("https://www.google.com",                    False, "google homepage"),
    ("https://docs.google.com/fake-page",         True,  "google docs abuse"),
    ("https://www.google.com/url?q=evil.com",     True,  "google redirect abuse"),
    ("https://sites.google.com/view/fake",        True,  "google sites abuse"),
    ("https://www.wikipedia.org",                 False, "wikipedia homepage"),
    ("https://www.microsoft.com",                 False, "microsoft homepage"),
    ("https://www.youtube.com",                   False, "youtube homepage"),
]

all_passed = True
for url, expected, label in tests:
    res    = check_phishtank_fixed(url, db_path)
    actual = res['is_phishing']
    passed = actual == expected
    if not passed:
        all_passed = False
    icon   = '' if passed else ''
    print(f"{icon} {label:<25}: {str(actual):<5} (expected {expected})")

print()
if all_passed:
    print(" ALL TESTS PASSED — Universal fix works for any domain!")
else:
    print("  Some tests failed — tell me which ones")

In [ ]:
# ============================================================
# FINAL CORRECT PhishTank checker
# Checks raw URL string — no parsing bug
# ============================================================

def check_phishtank_fixed(url, db_path=None):
    result = {
        'checked'    : False,
        'is_phishing': False,
        'error'      : None
    }

    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available"
        return result

    # Redirect abuse paths — checked against RAW url string
    # These mean a legitimate domain is being abused as redirect
    redirect_abuse_paths = [
        '/url?', '/amp/', '/amp/s/', '/amp/a/',
        '/redirect', '/l?', '/link?',
        '//amp'
    ]

    try:
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace('www.', '').strip()
        # Use raw URL for pattern matching — avoid urllib splitting bug
        url_raw    = url.lower()

        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '').strip()

                # Step 1: Domains must match exactly — always
                if url_domain != phish_domain:
                    continue

                # Step 2: Check if PhishTank entry is redirect abuse
                # Check DIRECTLY on raw URL string — no parsing
                phish_is_redirect = any(
                    pattern in phish_url
                    for pattern in redirect_abuse_paths
                )

                if phish_is_redirect:
                    # PhishTank entry is redirect abuse
                    # Only flag if OUR url also has redirect pattern
                    our_is_redirect = any(
                        pattern in url_raw
                        for pattern in redirect_abuse_paths
                    )
                    if our_is_redirect:
                        result['is_phishing'] = True
                        break
                else:
                    # Normal phishing domain — domain match enough
                    result['is_phishing'] = True
                    break

        result['checked'] = True

    except Exception as e:
        result['error'] = str(e)

    return result


# --- Run ALL tests ---
print("FINAL PHISHTANK TESTS")
print("=" * 55)

tests = [
    ("https://www.google.com",                    False, "google homepage      "),
    ("https://docs.google.com/fake-page",         True,  "google docs abuse    "),
    ("https://www.google.com/url?q=evil.com",     True,  "google redirect abuse"),
    ("https://sites.google.com/view/fake",        True,  "google sites abuse   "),
    ("https://www.wikipedia.org",                 False, "wikipedia homepage   "),
    ("https://www.microsoft.com",                 False, "microsoft homepage   "),
    ("https://www.youtube.com",                   False, "youtube homepage     "),
]

all_passed = True
for url, expected, label in tests:
    res    = check_phishtank_fixed(url, db_path)
    actual = res['is_phishing']
    passed = actual == expected
    if not passed:
        all_passed = False
    icon = '' if passed else ''
    print(f"{icon} {label}: {str(actual):<5} (expected {expected})")

print()
if all_passed:
    print(" ALL TESTS PASSED!")
    print("   PhishTank checker is working correctly.")
    print("   Ready to close Phase 4!")
else:
    print(" Still failing — paste output and I'll dig deeper")

In [ ]:
# ============================================================
# DEFINITIVE FINAL PhishTank checker — paste this as new cell
# Catches ALL redirect abuse patterns including query-based
# ============================================================

def check_phishtank_fixed(url, db_path=None):
    result = {
        'checked'    : False,
        'is_phishing': False,
        'error'      : None
    }

    if db_path is None or not os.path.exists(db_path):
        result['error'] = "PhishTank DB not available"
        return result

    # ALL redirect abuse patterns — path AND query based
    # Checked against raw URL string to avoid urllib parsing bug
    redirect_abuse_patterns = [
        '/url?', '/amp/', '/amp/s/', '/amp/a/',
        '/redirect', '/l?', '/link?', '//amp',
        '?url=', '&url=', '?q=', 'shortlink='
    ]

    try:
        parsed     = urllib.parse.urlparse(url)
        url_domain = parsed.netloc.lower().replace('www.', '').strip()
        url_raw    = url.lower()

        with open(db_path, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            for row in reader:
                phish_url    = row.get('url', '').lower()
                phish_parsed = urllib.parse.urlparse(phish_url)
                phish_domain = phish_parsed.netloc.lower().replace('www.', '').strip()

                # Step 1: Domains must match exactly
                if url_domain != phish_domain:
                    continue

                # Step 2: Is this PhishTank entry a redirect abuse?
                # Check directly on raw string — no urllib splitting
                phish_is_redirect = any(
                    pattern in phish_url
                    for pattern in redirect_abuse_patterns
                )

                if phish_is_redirect:
                    # Only flag if OUR url also uses redirect pattern
                    our_is_redirect = any(
                        pattern in url_raw
                        for pattern in redirect_abuse_patterns
                    )
                    if our_is_redirect:
                        result['is_phishing'] = True
                        break
                else:
                    # Normal phishing domain — domain match is enough
                    result['is_phishing'] = True
                    break

        result['checked'] = True

    except Exception as e:
        result['error'] = str(e)

    return result


# --- FINAL TEST ---
print("DEFINITIVE PHISHTANK TESTS")
print("=" * 55)

tests = [
    ("https://www.google.com",                False, "google homepage      "),
    ("https://docs.google.com/fake-page",     True,  "google docs abuse    "),
    ("https://www.google.com/url?q=evil.com", True,  "google redirect abuse"),
    ("https://sites.google.com/view/fake",    True,  "google sites abuse   "),
    ("https://www.wikipedia.org",             False, "wikipedia homepage   "),
    ("https://www.microsoft.com",             False, "microsoft homepage   "),
    ("https://www.youtube.com",              False,  "youtube homepage     "),
]

all_passed = True
for url, expected, label in tests:
    res    = check_phishtank_fixed(url, db_path)
    actual = res['is_phishing']
    passed = actual == expected
    if not passed:
        all_passed = False
    icon   = '' if passed else ''
    print(f"{icon} {label}: {str(actual):<5} (expected {expected})")

print()
if all_passed:
    print(" ALL 7 TESTS PASSED!")
    print("   PhishTank is working correctly for ALL domains.")
    print("   Ready to close Phase 4!")
else:
    print(" Still failing")

In [ ]:
# ============================================================
# FINAL PIPELINE TEST — 5 URLs through complete system
# Uses our definitive fixed PhishTank checker
# ============================================================

test_urls = [
    "https://tinyurl.com/wikipedia-en",
    "https://bit.ly/3google",
    "https://cutt.ly/github",
    "https://tinyurl.com/python-docs",
    "https://rb.gy/stackoverflow",
]

results_log = []

print("TESTING 5 URLs THROUGH COMPLETE FIXED PIPELINE")
print("=" * 60)

for url in test_urls:
    print(f"\n{'─'*60}")
    result = check_url_final(url, db_path=db_path, verbose=True)
    results_log.append({
        'Input URL'    : result['input_url'],
        'Expanded URL' : result['expanded_url'],
        'PhishTank'    : result['phishtank_result'],
        'ML Prob'      : f"{result['ml_result']['phishing_prob']}%"
                         if result.get('ml_result') else 'N/A',
        'Final Verdict': result['final_verdict'],
        'Reason'       : result['verdict_reason']
    })
    time.sleep(1)

print("\n\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
display(pd.DataFrame(results_log))